PMI-based similarity

In [1]:
#setup

from pathlib import Path
from collections import Counter, defaultdict
import math
import random
import re

import pandas as pd

pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("../data/Sentences_50Agree.txt")
OUTPUT_DIR = Path("../results/task4_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [3]:
#load dataset

rows = []

with open(DATA_PATH, "r", encoding="latin-1") as f:
    for line in f:
        line = line.strip()
        if line:
            text, label = line.rsplit("@", 1)
            rows.append({
                "text": text.strip(),
                "label": label.strip()
            })

df = pd.DataFrame(rows)

print("Dataset loaded successfully.")
print("Number of examples:", len(df))
print("Labels:", sorted(df["label"].unique()))

df.head()

Dataset loaded successfully.
Number of examples: 4846
Labels: ['negative', 'neutral', 'positive']


,text,label
0,"According to Gran , the company has no plans to move all production to Russia , although that is where the company i...",neutral
1,"Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies work...",neutral
2,The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; con...,negative
3,With the new production plant the company would increase its capacity to meet the expected increase in demand and wo...,positive
4,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth ...",positive


In [ ]:
from nltk.tokenize import RegexpTokenizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

tokenizer = RegexpTokenizer(r"[A-Za-z]+")

stop_words = set(ENGLISH_STOP_WORDS)

#same words i kept in preprocessing
words_to_keep = {
    "no", "not", "nor",
    "up", "down",
}

stop_words = stop_words - words_to_keep

def tokenize_pmi(text):
    tokens = tokenizer.tokenize(text.lower())
    
    tokens = [
        token for token in tokens
        if token not in stop_words
        and len(token) > 1
    ]
    
    return tokens

df["tokens"] = df["text"].apply(tokenize_pmi)

df[["text", "tokens"]].head()

,text,tokens
0,"According to Gran , the company has no plans to move all production to Russia , although that is where the company i...","[according, gran, company, no, plans, production, russia, company, growing]"
1,"Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies work...","[technopolis, plans, develop, stages, area, no, square, meters, order, host, companies, working, computer, technolog..."
2,The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; con...,"[international, electronic, industry, company, elcoteq, laid, tens, employees, tallinn, facility, contrary, earlier,..."
3,With the new production plant the company would increase its capacity to meet the expected increase in demand and wo...,"[new, production, plant, company, increase, capacity, meet, expected, increase, demand, improve, use, raw, materials..."
4,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth ...","[according, company, updated, strategy, years, basware, targets, long, term, net, sales, growth, range, operating, p..."


In [ ]:
# we keep only words that appear at least 5 times

MIN_WORD_FREQ = 5

all_tokens_before_filtering = [
    token
    for tokens in df["tokens_pmi"]
    for token in tokens
]

word_counts = Counter(all_tokens_before_filtering)

vocabulary = {
    word for word, freq in word_counts.items()
    if freq >= MIN_WORD_FREQ
}

corpus_tokens = []

for tokens in df["tokens_pmi"]:
    filtered_tokens = [
        token for token in tokens
        if token in vocabulary
    ]
    corpus_tokens.append(filtered_tokens)

all_tokens = [
    token
    for tokens in corpus_tokens
    for token in tokens
]

print("Minimum word frequency:", MIN_WORD_FREQ)
print("Total tokens before filtering:", len(all_tokens_before_filtering))
print("Vocabulary size before filtering:", len(word_counts))
print("Total tokens after filtering:", len(all_tokens))
print("Vocabulary size after filtering:", len(vocabulary))
print("Removed vocabulary items:", len(word_counts) - len(vocabulary))

Minimum word frequency: 5
Total tokens before filtering: 61137
Vocabulary size before filtering: 8992
Total tokens after filtering: 50364
Vocabulary size after filtering: 2105
Removed vocabulary items: 6887


In [ ]:
#co occurrence counts

WINDOW_SIZE = 1

cooccurrence_counts = Counter()
target_counts = Counter()
context_counts = Counter()

for tokens in corpus_tokens:
    for i, target_word in enumerate(tokens):
        left = max(0, i - WINDOW_SIZE)
        right = min(len(tokens), i + WINDOW_SIZE + 1)
        
        for j in range(left, right):
            if i == j:
                continue
            
            context_word = tokens[j]
            
            cooccurrence_counts[(target_word, context_word)] += 1
            target_counts[target_word] += 1
            context_counts[context_word] += 1

total_cooccurrences = sum(cooccurrence_counts.values())

print("Number of co-occurrence pairs:", len(cooccurrence_counts))
print("Total co-occurrences:", total_cooccurrences)

Number of co-occurrence pairs: 55455
Total co-occurrences: 91046


PMI(w,c) = log2( P(w,c) / (P(w) P(c)) )

PMI(w,c) = log2( count(w,c) * total / (count(w) * count(c)) )

In [ ]:
#pmi vectors

pmi_vectors = defaultdict(dict)

for (target_word, context_word), cooc_count in cooccurrence_counts.items():
    numerator = cooc_count * total_cooccurrences
    denominator = target_counts[target_word] * context_counts[context_word]
    
    pmi = math.log2(numerator / denominator)
    
    pmi_vectors[target_word][context_word] = pmi

print("Number of PMI vectors:", len(pmi_vectors))

Number of PMI vectors: 2105


In [ ]:
def vector_norm(vector):
    return math.sqrt(sum(value ** 2 for value in vector.values()))

#vector cosine similarity
def cosine_similarity_sparse(vector_a, vector_b):
    if not vector_a or not vector_b:
        return 0.0
    
    norm_a = vector_norm(vector_a)
    norm_b = vector_norm(vector_b)
    
    if norm_a == 0 or norm_b == 0:
        return 0.0
    
    if len(vector_a) > len(vector_b):
        vector_a, vector_b = vector_b, vector_a
    
    dot_product = 0.0
    
    for key, value in vector_a.items():
        if key in vector_b:
            dot_product += value * vector_b[key]
    
    return dot_product / (norm_a * norm_b)

#finds most similar words based on cosine similarity
def most_similar_words(target_word, pmi_vectors, top_k=5):
    if target_word not in pmi_vectors:
        return []
    
    target_vector = pmi_vectors[target_word]
    similarities = []
    
    for other_word, other_vector in pmi_vectors.items():
        if other_word == target_word:
            continue
        
        similarity = cosine_similarity_sparse(target_vector, other_vector)
        
        similarities.append((other_word, similarity))
    
    similarities = sorted(similarities, key=lambda x: x[1], reverse=True)
    
    return similarities[:top_k]


In [28]:
#random words 

MIN_CONTEXTS = 5
N_RANDOM_WORDS = 10

candidate_words = [
    word for word, vector in pmi_vectors.items()
    if len(vector) >= MIN_CONTEXTS
]

print("Number of candidate words:", len(candidate_words))

random_words = random.sample(candidate_words, N_RANDOM_WORDS)

random_words

Number of candidate words: 2091


['initial',
 'chemical',
 'september',
 'terminal',
 'newspapers',
 'ordered',
 'pohjola',
 'dongguan',
 'raised',
 'suite']

In [ ]:
#most similar words for each random word

TOP_K = 5

similarity_results = []

for target_word in random_words:
    similar_words = most_similar_words(target_word, pmi_vectors, top_k=TOP_K)
    
    similarity_results.append({
        "target_word": target_word,
        "most_similar_words": ", ".join([word for word, score in similar_words]),
        "similarity_scores": ", ".join([f"{score:.3f}" for word, score in similar_words])
    })

similarity_results_df = pd.DataFrame(similarity_results)

similarity_results_df

,target_word,most_similar_words,similarity_scores
0,initial,"tender, symbian, preliminary, implementing, confirmed","0.219, 0.197, 0.178, 0.160, 0.147"
1,chemical,"ministry, consulting, poyry, consumption, forestry","0.211, 0.188, 0.176, 0.170, 0.156"
2,september,"october, july, august, february, april","0.255, 0.217, 0.202, 0.194, 0.191"
3,terminal,"hamina, de, cargo, vendors, commissioned","0.278, 0.209, 0.185, 0.146, 0.143"
4,newspapers,"handset, sami, juha, hardware, samsung","0.201, 0.196, 0.180, 0.161, 0.151"
5,ordered,"cruise, friendly, targeting, lng, passengers","0.236, 0.142, 0.142, 0.125, 0.121"
6,pohjola,"deutsche, sto, sampo, syndicated, op","0.208, 0.190, 0.179, 0.178, 0.165"
7,dongguan,"southern, partly, noted, running, petersburg","0.193, 0.191, 0.190, 0.181, 0.163"
8,raised,"kci, counter, surged, correspond, benefon","0.211, 0.194, 0.182, 0.163, 0.160"
9,suite,"stonegate, motorola, gypsii, gps, mail","0.339, 0.248, 0.215, 0.215, 0.215"


In [34]:
similarity_results_path = OUTPUT_DIR / "pmi_similarity_results.csv"
similarity_results_df.to_csv(similarity_results_path, index=False)
print("Saved summary results to:", similarity_results_path)


Saved summary results to: ..\results\task4_outputs\pmi_similarity_results.csv
